In [1]:
import qutip as qt
from qutip import tensor, basis, qeye
import numpy as np
from quantum_logical.trotter_diff import Trotterization
from tqdm import tqdm

# there is circular importation that needs fixed 

In [2]:
from quantum_logical.state import state as st

In [3]:
def gate(dim, N):
    from quantum_logical.gate_extender import Gate_extender, Convert_levels
    from quantum_logical.cnot_gate_creation import cnot
    # creating the gates
    dim = dim
    N = N
    # creating the set of cnots (this will operate between the g and e levels)

    cnot1 = cnot(N=N, target=3, control=0, high=2, low=0)
    cnot2 = cnot(N=N, target=3, control=1, high=2, low=0)
    
    cnot3 = cnot(N=N, target=4, control=1, high=2, low=0)
    cnot4 = cnot(N=N, target=4, control=2, high=2, low=0)

    # the x_gate needs to be made in a qutrit gate and will involve conversion 
    x_gate = qt.Qobj([[0, 1],[1, 0]])

    hada = qt.Qobj([[1/np.sqrt(2), 0, 1/np.sqrt(2)], [0, 1, 0], [1/np.sqrt(2), 0, -1/np.sqrt(2)]])

    gate_extention = Gate_extender(num_qubits=1)
    x_gate = gate_extention.qubit_to_qudit(gate=x_gate, from_dim=2, to_dim=dim)


    # conversion of some of the gates into the qutrit space 
    new_dim = dim
    converter = Convert_levels(num_qubits=1)
    x_gate = converter.level_conversion(levels=[0,2], dim=dim, gate=x_gate, qubits=None)


    x_layer = tensor(tensor([x_gate] * 3), tensor([qeye(new_dim)] * 2))
    hada_layer = tensor(tensor([hada] * 3), tensor([qeye(new_dim)] * 2))

    # building the correction z_gate 
    correction_x = qt.Qobj([[0, 1],[1, 0]])
    gate_extention = Gate_extender(num_qubits=1)
    correction_x = gate_extention.qubit_to_qudit(gate=correction_x, from_dim=2, to_dim=3)
    converter = Convert_levels(num_qubits=1)
    correction_x = converter.level_conversion(levels=[1,2], dim=new_dim, gate=correction_x, qubits=None)

    correction_z = (hada * correction_x * hada.dag())


    cnots = [cnot1, cnot2, cnot3, cnot4]
    return cnots, correction_z, hada_layer, x_layer

In [4]:
def repetition_correction(values, rho_initial, total_time):

    cnots = values[0]
    dim = values[2]
    N = values[3]
    x_layer = values[4]
    correction_z = values[5]
    hada_layer = values[6]
    T = values[1]

    cnot3 = hada_layer * cnots[0] * hada_layer.dag()
    cnot4 = hada_layer * cnots[1] * hada_layer.dag()
    cnot5 = hada_layer * cnots[2] * hada_layer.dag()
    cnot6 = hada_layer * cnots[3] * hada_layer.dag()

    initial_state = qt.ptrace(rho_initial, [0,1,2])
    ancilla_states = tensor(basis(dim, 0), basis(dim, 0)) * tensor(basis(dim, 0), basis(dim, 0)).dag()
    full_initial_state = tensor(initial_state, ancilla_states)

    # stored information
    states = []

    def neilson_fid(rho, sigma):
        return (((rho.sqrtm()) * sigma * (rho.sqrtm())).sqrtm()).tr()


    # gate setups 
    # gates = [[x_layer], [cnot3], [cnot4], [cnot5], [cnot6], [x_layer]]
    gates = [[hada_layer], [cnot3], [cnot4], [cnot5], [cnot6], [hada_layer]]
    cnot_time = .5
    gate_times = [.03, cnot_time, cnot_time, cnot_time, cnot_time, .03]
    # gate_times = [cnot_time, cnot_time, cnot_time, cnot_time]


    # stabilizer extraction
    # rho_encoded = hada_layer * full_initial_state * hada_layer.dag()
    # trotter_dt = .03 / 20
    # trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=N, qudit="qutrit")
    # rho_evo = trotter.apply(rho=rho_encoded, duration=.03, unitary=[x_layer], errors=True, amp=True, dephasing=True)
    # states.extend(rho_evo)
    # rho_encoded = rho_evo[-1]
    # rho_encoded = hada_layer * rho_encoded * hada_layer.dag()
    # total_time += .03
    # rho_encoded = hada_layer * full_initial_state * hada_layer.dag()
    rho_encoded = full_initial_state 


    for i in range(len(gates)):
        trotter_dt = gate_times[i] / 10
        trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=N, qudit="qutrit")
        rho_evo = trotter.apply(rho=rho_encoded, duration=gate_times[i], unitary=gates[i], errors=True, amp=True, dephasing=True, hada_=True)
        states.extend(rho_evo)
        rho_encoded = rho_evo[-1]
        total_time += gate_times[i]


    # rho_encoded = hada_layer * rho_encoded * hada_layer.dag()
    # trotter_dt = .03 / 20
    # trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=N, qudit="qutrit")
    # rho_evo = trotter.apply(rho=rho_encoded, duration=.03, unitary=[x_layer], errors=True, amp=True, dephasing=True)
    # states.extend(rho_evo)
    # rho_encoded = rho_evo[-1]
    # total_time += .03

        
    # states.append(hada_layer * rho_encoded * hada_layer.dag())
    states.append(rho_encoded)


    # projection operators  
    proj = [qt.tensor(qt.qeye(dim), qt.qeye(dim), qt.qeye(dim), 
            (qt.tensor(qt.basis(dim, i), qt.basis(dim, j)) * (qt.tensor(qt.basis(dim, i), qt.basis(dim, j))).dag())) 
            for i in range(3) for j in range(3)]


    # correction_operators
    r00 = r01 = r10 = r11 = r12 = r21 = qt.tensor([qt.qeye(dim)] * 5)
    r02 = qt.tensor(qt.qeye(dim), qt.qeye(dim), correction_z, qt.tensor([qt.qeye(dim)] * 2))
    r20 = qt.tensor(correction_z, qt.tensor([qt.qeye(dim)] * 4))
    r22 = qt.tensor(qt.qeye(dim), correction_z, qt.qeye(dim), qt.tensor([qt.qeye(dim)] * 2))
    recovery_ops = [[r00], [r01], [r02], [r10], [r11], [r12], [r20], [r21], [r22]]

    # # Correction operators 
    # r00 = qt.tensor([qt.qeye(dim)] * 5)
    # r01 = qt.tensor(qt.qeye(dim), qt.qeye(dim), correction_z, qt.tensor([qt.qeye(dim)] * 2))
    # r10 = qt.tensor(correction_z, qt.tensor([qt.qeye(dim)] * 4))
    # r11 = qt.tensor(qt.qeye(dim), correction_z, qt.qeye(dim), qt.tensor([qt.qeye(dim)] * 2))
    # recovery_ops = [[r00], [r01], [r10], [r11]]

    # measurement 
    measured_states = []
    measurement_duration = 2
    trotter_dt = measurement_duration / 20
    trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=N, qudit="qutrit")
    measurement_delay_evo = trotter.apply(rho=states[-1], duration=measurement_duration, unitary=[tensor([qt.qeye(dim)] * N)], errors=True, amp=True, dephasing=True)
    measured_states.append(measurement_delay_evo)
    proj_results_after_measurement = [(measurement_delay_evo[-1] * proj).tr() for proj in proj]
    proj_states_after_measurment = [hada_layer * (proj * measurement_delay_evo[-1] * proj.dag()) * hada_layer.dag() for proj in proj]
    total_time += measurement_duration


    corrected_states = []
    recovery_duration = .03
    for i in range(len(recovery_ops)):
        if proj_results_after_measurement[i] != 0:
            trotter_dt = recovery_duration / 20
            trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=N, qudit="qutrit")
            corrected_state = trotter.apply(proj_states_after_measurment[i], duration=recovery_duration, unitary=recovery_ops[i], errors=True, amp=True, dephasing=True)
            corrected_states.append(corrected_state[-1])
        else:
            corrected_states.append(recovery_ops[i][0] * proj_states_after_measurment[i] * recovery_ops[i][0].dag())
    total_time += recovery_duration


    # combining the branches 
    repetition_corrected_state = sum([proj_results_after_measurement[j] * corrected_states[j] for j in range(len(proj_results_after_measurement))])
    # repetition_corrected_state = hada_layer * repetition_corrected_state * hada_layer.dag()

    return repetition_corrected_state, total_time, proj_results_after_measurement

In [5]:
# erasure circuit 
def erasure_correction(rho, N, T, total_time):
    from quantum_logical.cnot_gate_creation import cnot

    dim = 3
    initial_state = qt.ptrace(rho, [0,1,2])
    ancilla_states = tensor(basis(dim, 0), basis(dim, 0), basis(dim, 0)) * tensor(basis(dim, 0), basis(dim, 0), basis(dim, 0)).dag()
    full_initial_state = tensor(initial_state, ancilla_states)

    hada = qt.Qobj([[1/np.sqrt(2), 0, 1/np.sqrt(2)], [0, 1, 0], [1/np.sqrt(2), 0, -1/np.sqrt(2)]])
    hada_layer = tensor(tensor([hada] * 3), tensor([qeye(dim)] * 3))

    # detection gate creation
    # cnot_create = Convert_levels(num_qubits=N)
    # cnot1 = cnot_create.Cnot(dim=3, target=3, control=0, high=1, low=0)
    cnot1 = hada_layer * cnot(N=N, target=3, control=0, high=1, low=0) * hada_layer.dag()
    # cnot2 = cnot_create.Cnot(dim=3, target=4, control=1, high=1, low=0)
    cnot2 = hada_layer * cnot(N=N, target=4, control=1, high=1, low=0) * hada_layer.dag()
    # cnot3 = cnot_create.Cnot(dim=3, target=5, control=2, high=1, low=0)
    cnot3 = hada_layer * cnot(N=N, target=5, control=2, high=1, low=0) * hada_layer.dag()



    gates = [cnot1, cnot2, cnot3]
    cnot_time = .5
    gate_times = [cnot_time for _ in range(len(gates))]


    # running the circuit 
    # rho_encoded = hada_layer * full_initial_state * hada_layer.dag()
    rho_encoded = full_initial_state


    for i in range(len(gates)):
        trotter_dt = gate_times[i] / 10
        trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=N, qudit="qutrit")
        rho_evo = trotter.apply(rho=rho_encoded, duration=gate_times[i], unitary=[gates[i]], errors=True, amp=True, dephasing=True)
        rho_encoded = rho_evo[-1]
        total_time += gate_times[i]


    # measurement
    # measurement operators 
    proj = [tensor(qeye(dim), qeye(dim), qeye(dim), tensor(basis(dim, i), basis(dim, j), basis(dim, k))  * tensor(basis(dim, i), basis(dim, j), basis(dim, k)).dag()) 
                for i in [0,1] for j in [0,1] for k in [0,1]]

    measurement_duration = 2
    trotter_dt = measurement_duration / 10
    trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=N, qudit="qutrit")
    total_time += measurement_duration

    measurement_evo = trotter.apply(rho=rho_encoded, duration=measurement_duration, unitary=[tensor([qeye(dim)] * N)], errors=True, amp=True, dephasing=True)
    # state_an = hada_layer * measurement_evo[-1] * hada_layer.dag()

    # projection results
    proj_res = [(proj * measurement_evo[-1]).tr() for proj in proj]
    proj_states = [(proj * measurement_evo[-1] * proj.dag()) for proj in proj]


    # Correction based on the results 
    # correction gates
    # cnot_create = Convert_levels(num_qubits=N)
    # cnot1 = cnot_create.Cnot(dim=3, target=2, control=0, high=2, low=1)
    cnot1 = hada_layer * cnot(N=N, target=2, control=0, high=2, low=1) * hada_layer.dag()
    cnot1 = cnot(N=N, target=2, control=0, high=2, low=1) 
    # cnot2 = cnot_create.Cnot(dim=3, target=1, control=0, high=2, low=1)
    cnot2 = hada_layer * cnot(N=N, target=1, control=0, high=2, low=1) * hada_layer.dag()
    cnot2 = cnot(N=N, target=1, control=0, high=2, low=1) 
    # cnot3 = cnot_create.Cnot(dim=3, target=0, control=1, high=2, low=1)
    cnot3 = hada_layer * cnot(N=N, target=0, control=1, high=2, low=1) * hada_layer.dag()
    cnot3 = cnot(N=N, target=0, control=1, high=2, low=1)
    # cnot4 = cnot_create.Cnot(dim=3, target=2, control=1, high=2, low=1)
    cnot4 = hada_layer * cnot(N=N, target=2, control=1, high=2, low=1) * hada_layer.dag()
    cnot4 = cnot(N=N, target=2, control=1, high=2, low=1)
    # cnot5 = cnot_create.Cnot(dim=3, target=0, control=2, high=2, low=1)
    cnot5 = hada_layer * cnot(N=N, target=0, control=2, high=2, low=1) * hada_layer.dag()
    cnot5 = cnot(N=N, target=0, control=2, high=2, low=1)
    # cnot6 = cnot_create.Cnot(dim=3, target=1, control=2, high=2, low=1)
    cnot6 = hada_layer * cnot(N=N, target=1, control=2, high=2, low=1) * hada_layer.dag()
    cnot6 = cnot(N=N, target=1, control=2, high=2, low=1)

    # correction_operators
    r000 = [qt.tensor([qt.qeye(dim)] * N)]
    r001 = [cnot1]
    r010 = [cnot2]
    r011 = [cnot2, cnot1]
    r100 = [cnot3]
    r101 = [cnot3, cnot4]
    r110 = [cnot5, cnot6]
    r111 = [qt.tensor([qt.qeye(dim)] * N)]

    recovery_ops = [r000, r001, r010, r011, r100, r101, r110, r111]


    # start the correction procedure 
    # correction_cycle 
    correction_duration = .5
    corrected_states = []

    for i in range(len(recovery_ops)):
        if proj_res[i] != 0:
            trotter_dt = correction_duration / 10
            trotter = Trotterization(trotter_dt=trotter_dt, T1=T[0], T2=T[1], dim=dim, num_qubits=N, qudit="qutrit")

            corrected_state = trotter.apply(rho=hada_layer * proj_states[i] * hada_layer.dag(), duration=correction_duration, unitary=recovery_ops[i], errors=True, amp=True, dephasing=True)
            corrected_states.append(corrected_state)
        else:
            corrected_states.append([recovery_ops[i][0] * (hada_layer * proj_states[i] * hada_layer.dag()) * (recovery_ops[i][0]).dag()] * int(correction_duration/ trotter_dt))

    total_time += correction_duration
    # combining the corrected states
    erasure_corrected_state = sum([proj_res[j] * corrected_states[j][-1] for j in range(len(proj_res))])

    erasure_corrected_state = hada_layer * erasure_corrected_state * hada_layer.dag()

    return erasure_corrected_state, total_time

In [6]:
N = 5
dim = 3
cnots, correction_z, hada_layer, x_layer = gate(dim=dim, N=N)

In [7]:
from quantum_logical.state import state as st

In [8]:
hada = qt.Qobj([[1/np.sqrt(2), 0, 1/np.sqrt(2)], [0, 1, 0], [1/np.sqrt(2), 0, -1/np.sqrt(2)]])
# vector setup 
basis0 = qt.Qobj([[1],[0],[0]])
basis1 = qt.Qobj([[0],[1],[0]])
basis2 = qt.Qobj([[0],[0],[1]])
vector0 = hada * basis0
vector1 = hada * basis1
vector2 = hada * basis2
vectors = [vector0, vector1, vector2]
vectors = [tensor(i,j,k) for i in vectors for j in vectors for k in vectors]

In [9]:
iterations = 1
t1_list = np.linspace(50, 160, iterations)
t_list = []
for i in range(len(t1_list)):
    # t2s = np.linspace((t1_list[i] * (2/3)), (t1_list[i] * (2/3)), 1)
    t2s = np.linspace(t1_list[i] * (2/3), 2* t1_list[i] - 1, 1)
    for j in range(len(t2s)):
        t_list.append([t1_list[i], t2s[j]])

values = []
for i in range(iterations):
    values.append([cnots, t_list[i], dim, N, x_layer, correction_z, hada_layer])

physical_error = []
logical_error = []
phase_errors = []
erasure_errors = []
t1 = []
t2 = []
physical_err = []

cycles = 1

state_choices = [[["-", "-", "-"], 1, 0], [["+", "+", "+"], 1, 0], 
                     [["+", "+", "+"], 1, 1], [["+", "+", "+"], 1, -1], 
                     [["+", "+", "+"], 1, 1j], [["+", "+", "+"], 1, -1j]]
# state_choices = [[["-", "-", "-"], 1, 0]]

proj_res = []

for state_choice in state_choices:
    log_err = []
    phys_err = []
    time = []
    proj_res_ = []
    eras_err = []
    phase_err = []
    physical_rate = []


    # this has to be the ugliest code you have ever written (fix this)
    if state_choice == [["-", "-", "-"], 1, 0]:
        vectors_phase = []
        vectors_erasure = []
        rho, state = st(qubit_choices=["-", "-", "+"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_phase.append(state)
        rho, state = st(qubit_choices=["-", "+", "-"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_phase.append(state)
        rho, state = st(qubit_choices=["+", "-", "-"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_phase.append(state)
        rho, state = st(qubit_choices=["-", "-", "1"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["-", "1", "1"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["1", "-", "-"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["1", "-", "1"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["1", "1", "-"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["-", "1", "-"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
    elif state_choice[0] == ["+", "+", "+"]:
        vectors_phase = []
        rho, state = st(qubit_choices=["-", "+", "+"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_phase.append(state)
        rho, state = st(qubit_choices=["+", "+", "-"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_phase.append(state)
        rho, state = st(qubit_choices=["+", "-", "+"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_phase.append(state)
        rho, state = st(qubit_choices=["+", "+", "1"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["+", "1", "1"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["1", "+", "+"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["1", "+", "1"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["1", "1", "+"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)
        rho, state = st(qubit_choices=["+", "1", "+"], dim=3, alpha=state_choice[1], beta=state_choice[2])
        vectors_erasure.append(state)

    for value in tqdm(values):
    
        rho_encoded, state_vector_ = st(qubit_choices=state_choice[0], dim=3, alpha=state_choice[1], beta=state_choice[2])

        order = [0]
        # order = [0]
        total_time = 0
        for i in range(cycles):
            for choice in order:
                if choice == 0:
                    # rho_encoded, total_time, projection_results = repetition_correction(values=value, rho_initial=rho_encoded, total_time=total_time)
                    rho_encoded, total_time, projections = repetition_correction(values=value, rho_initial=rho_encoded, total_time=total_time)


                elif choice == 1:
                    # rho_encoded, total_time = erasure_correction(rho=rho_encoded, N=6, T=value[1], total_time=total_time)
                    rho_encoded, total_time = erasure_correction(rho=rho_encoded, N=6, T=value[1], total_time=total_time)




        def neilson_fid(rho, sigma):
            return (((rho.sqrtm()) * sigma * (rho.sqrtm())).sqrtm()).tr()
        

        # proj_res_.append(1 - projections[0])

        logical = [state_vector_]
        vals = []
        for vec in logical:
            val = (vec.dag() * qt.ptrace(rho_encoded, [0,1,2]) * vec)[0][0][0]
            vals.append(np.abs(val))
        log_err.append(np.abs(1 - np.abs(sum(vals))))

        vals = []
        for vec in vectors_phase:
            val = (vec.dag() * qt.ptrace(rho_encoded, [0,1,2]) * vec)[0][0][0]
            vals.append(np.abs(val))
        phase_err.append(np.abs(sum(vals)))

        vals = []
        for vec in vectors_erasure:
            val = (vec.dag() * qt.ptrace(rho_encoded, [0,1,2]) * vec)[0][0][0]
            vals.append(np.abs(val))
        eras_err.append(np.abs(sum(vals)))

        vals = []
        for vec in vectors:
            val = (vec.dag() * qt.ptrace(rho_encoded, [0,1,2]) * vec)[0][0][0]
            vals.append(np.abs(val))

        t_phase = (2 * value[1][0] * value[1][1])/(2 * value[1][0] - value[1][1])
        physical_error_val = ((1 - np.exp((-total_time) * ((1/value[1][0])))) + (1 - np.exp((-total_time) * ((1/t_phase)))) 
                              - (1 - np.exp((-total_time) * ((1/value[1][0])))) * (1 - np.exp((-total_time) * ((1/t_phase)))))
        phys_err.append(np.abs(physical_error_val))
        time.append(total_time)
        phys_rate = 1/(value[1][0]) + 1/((2*value[1][0]*value[1][1])/(2*value[1][0] - value[1][1]))
        physical_rate.append(phys_rate)


    phase_errors.append(phase_err)
    erasure_errors.append(eras_err)
    # proj_res.append(proj_res_)
    physical_error.append(phys_err)
    logical_error.append(log_err)

        

  0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\girgi\Desktop\Github\quantum_logical\venv\lib\site-packages\scipy\sparse\linalg\_onenormest.py:155: RuntimeWarning: invalid value encountered in divide
  Y /= np.abs(Y)


In [ ]:
logical_error

[[0.02696523815244245],
 [0.025780596119865384],
 [0.02855978260226899],
 [0.028559849126446935],
 [0.028559837021435053],
 [0.0285598370214335]]

In [10]:
# sort data
phys = []
logic = []
meas_error = []
e_errors = []
p_errors = []

for i in range(len(values)):
    phys.append(sum([physical_error[j][i] for j in range(len(state_choices))]) / len(state_choices))
    logic.append(sum([logical_error[j][i] for j in range(len(state_choices))]) / len(state_choices))
    meas_error.append(sum([proj_res[j][i] for j in range(len(state_choices))]) / len(state_choices))
    e_errors.append(sum([erasure_errors[j][i] for j in range(len(state_choices))]) / len(state_choices))
    p_errors.append(sum([phase_errors[j][i] for j in range(len(state_choices))]) / len(state_choices))

In [11]:
import csv

# Writing the arrays to a CSV file
with open('5th_order_codeword_state_phase_only_.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(phys)  
    writer.writerow(logic)
    writer.writerow(meas_error)
    writer.writerow(e_errors)  
    writer.writerow(p_errors)   
    writer.writerow(physical_rate)   
